# 카테고리 실험 템플릿

`CATEGORY` 와 `OPTIONS` 만 바꿔서 위에서부터 차례로 실행하면 됨.
여기서는 아무것도 저장하지 않으므로 마음껏 돌려도 됨. 확정한 옵션만 `src/model.py` 의 `CONFIGS` 에 적을 것.
자세한 순서는 `README_TEAM.md` 참고.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "example" else Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score

from src import Dataset
from src.evaluate import report, run
from src.model import CONFIGS, Model, foreground

plt.rcParams["font.family"] = "Malgun Gothic"      # 맥은 "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

## 1. 카테고리와 옵션

In [ ]:
CATEGORY = "mech_04"
OPTIONS = dict(CONFIGS.get(CATEGORY, {}))          # 설정표의 현재 값에서 시작
# OPTIONS.update(fg_thr=12)                        # 바꿔 볼 옵션. 한 번에 하나만: fg_thr, aug, top, sigma, bank_size

cat = Dataset().category(CATEGORY)
print(cat, OPTIONS)

## 2. 전경 마스크 확인 (모델을 돌리기 전에)

- 물체가 전부 흰색이어야 함. 물체 일부가 검게 빠지면 `fg_thr` 을 낮출 것.
- 배경의 반사광이 흰색으로 들어오면 `fg_thr` 을 높일 것.
- 아래 숫자는 val 결함이 마스크 안에 들어온 비율. 1.00 에 가까워야 함. 낮으면 그 결함은 절대 못 잡음.

In [ ]:
fg_thr = OPTIONS.get("fg_thr", 20)
x = cat.images("train", 8)
fg = foreground(x, fg_thr)
fig, ax = plt.subplots(2, 8, figsize=(20, 5))
for i in range(8):
    ax[0, i].imshow(x[i]); ax[1, i].imshow(fg[i], cmap="gray", vmin=0, vmax=1)
    ax[0, i].axis("off"); ax[1, i].axis("off")
plt.suptitle(f"{CATEGORY}  fg_thr={fg_thr}"); plt.tight_layout(); plt.show()

v, m = cat.images("val"), cat.masks("val")
bad = m.sum((1, 2)) > 0
big = np.repeat(np.repeat(foreground(v[bad], fg_thr), 8, 1), 8, 2)
cov = (m[bad] * big).sum((1, 2)) / m[bad].sum((1, 2))
print(f"결함이 마스크 안에 들어온 비율: 평균 {cov.mean():.2f}, 최저 {cov.min():.2f}, 10% 넘게 잘린 사진 {(cov < 0.9).sum()} / {bad.sum()}")

## 3. 학습하고 채점 (몇 분 걸림)

In [ ]:
res, maps, thr = run(lambda: Model(**OPTIONS), cat)
masks, images = cat.masks(res), cat.images(res)
r = report(res, masks, maps)
print({k: round(float(v), 3) for k, v in r.items()})

## 4. 어디가 약한가

- AUROC 가 낮다 → 특징이 불량을 구분하지 못함
- AUROC 는 높은데 아래 표의 F1 이 전부 낮다 → 임계값 문제 (모델은 그대로 두고 `team.py cv` 로 해결)
- AUROC 는 높은데 AUPRO 가 낮다 → 맵 문제 (전경 마스크, `sigma`)

유형별 장수가 적으니 0.05 정도의 차이는 무시할 것.

In [ ]:
normal = res.score[res.is_anomaly == 0]
rows = [(lab, len(g), roc_auc_score(np.r_[np.zeros(len(normal)), np.ones(len(g))], np.r_[normal, g.score]),
         g.mask_area_pct.median(), g.score.median())
        for lab, g in res[res.is_anomaly == 1].groupby("label")]
display(pd.DataFrame(rows, columns=["유형", "장수", "AUROC", "면적 중앙값 %", "점수 중앙값"]).round(3))
print(f"정상 점수: 중앙값 {normal.median():.1f}, 상위 5% {normal.quantile(.95):.1f}, 최고 {normal.max():.1f}")

display(pd.DataFrame([(q, normal.quantile(q), f1_score(res.is_anomaly, res.score >= normal.quantile(q)))
                      for q in (0.99, 0.95, 0.90, 0.85, 0.80, 0.70)],
                     columns=["정상 분위수", "임계값", "F1"]).round(3))

## 5. 맵으로 원인 보기

점수가 가장 높은 정상 3장(오탐 후보)과 가장 낮은 불량 3장(미탐)을 **같은 색 눈금**으로 그림.

| 보이는 것 | 뜻 | 해 볼 것 |
|---|---|---|
| 정상 사진의 배경이 밝다 | 배경이 점수를 올림 | `fg_thr` 조정 |
| 정상 사진의 물체 위(반사광, 끝부분)가 밝다 | 드문 정상이 뱅크에 없음 | `aug=True`(방향이 자유로운 물체), `bank_size` 키우기 |
| 불량 사진에서 결함 자리가 안 밝다 | 특징이 결함을 못 봄 | `sigma`, `top` 줄이기. 안 되면 새 기능이 필요하니 통합 담당에게 |

In [ ]:
order = res.score.to_numpy().argsort()
is_bad = res.is_anomaly.to_numpy() == 1
picks = [("오탐 후보", i) for i in order[::-1] if not is_bad[i]][:3] + [("미탐", i) for i in order if is_bad[i]][:3]
vmax = np.quantile(maps, 0.999)
fig, ax = plt.subplots(3, 6, figsize=(21, 10.5))
for j, (kind, i) in enumerate(picks):
    ax[0, j].imshow(images[i]); ax[0, j].set_title(f"{kind}  {res.path.iloc[i].split('val/')[-1]}
점수 {res.score.iloc[i]:.1f} / 임계값 {thr:.1f}", fontsize=9)
    ax[1, j].imshow(masks[i], cmap="gray", vmin=0, vmax=1); ax[1, j].set_title("정답", fontsize=9)
    ax[2, j].imshow(images[i]); ax[2, j].imshow(maps[i], cmap="inferno", alpha=0.6, vmin=0, vmax=vmax); ax[2, j].set_title("이상 맵 (공통 눈금)", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 6. 기록

- 옵션을 바꿨으면 3번부터 다시 실행하고, 바꾸기 전과 후의 AUROC / AUPRO 를 공유 기록표에 한 줄 적을 것 (효과가 없었어도 적음).
- 채택한 옵션은 `src/model.py` 의 `CONFIGS` 에서 **자기 카테고리 줄만** 고쳐서 커밋.